# SHEESHA - Wan2.1 text-to-video on a free Colab T4

Generates the picture clips for one episode. Free: a Colab free-tier T4 has 15 GB
of VRAM, which holds Wan2.1-T2V-1.3B. Your local GTX 1650 has 4 GB and cannot run it.

## How to use

The manifest decides how much work this does. It reads whatever jobs it finds:

* **1 job** = the free rehearsal. `python pipeline.py video-jobs 1 --test` writes
  this. One clip, ~30 min. Judge the look from that before spending time or money.
* **7 jobs** = the full episode. `python pipeline.py video-jobs 1` writes that.

Either way the steps are the same:

1. On your PC, run `python pipeline.py video-jobs 1` (add `--test` for one clip).
2. Go to drive.google.com, create a folder named `sheesha` inside `MyDrive`,
   and upload `video_jobs.json` into it. Path must be `MyDrive/sheesha/video_jobs.json`.
3. Open this notebook in Colab. **Runtime > Change runtime type > T4 GPU**.
4. Run every cell, top to bottom.
5. Download `MyDrive/sheesha/clips.zip` and unzip it over
   `yt_horror/render/clips/wan/` on your PC.
6. On your PC: `python pipeline.py preview 1` for a rehearsal,
   or `python pipeline.py episode 1 --provider wan` for the full episode.

## Before you start

* **Runtime > Change runtime type > T4 GPU.** Free tier is enough.
* Expect roughly **25-35 minutes per 5-second clip** on a T4.
* Free sessions disconnect if the tab sits idle for a while. Keep it focused
  and don't start with the laptop asleep. The loop is resumable: any clip
  already on disk is skipped, so a disconnect costs only the clip in progress.
* The first cell downloads ~9 GB of weights. That takes a few minutes.
* 480x832 is portrait 480p. The editor upscales to 1080x1920 locally.

In [ ]:
!pip install -q --upgrade "diffusers>=0.33" transformers accelerate imageio imageio-ffmpeg ftfy sentencepiece

In [ ]:
import json, os, time, pathlib, shutil
import torch
from google.colab import drive

drive.mount('/content/drive')

ROOT = pathlib.Path('/content/drive/MyDrive/sheesha')
JOBS = ROOT / 'video_jobs.json'
CLIPS = ROOT / 'clips'
CLIPS.mkdir(parents=True, exist_ok=True)

if not JOBS.exists():
    raise FileNotFoundError(
        f'{JOBS} not found. Run `python pipeline.py video-jobs 1` on your PC and\n'
        'upload episodes/ep01/video_jobs.json to MyDrive/sheesha/.')

manifest = json.loads(JOBS.read_text(encoding='utf-8'))
jobs = manifest['jobs']

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
vram = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
print(f'GPU      : {gpu}  ({vram:.1f} GB)')
print(f'episode  : {manifest["episode"]}   jobs: {len(jobs)}')
print(f'output   : {CLIPS}')

if 'T4' not in gpu:
    print('\nWARNING: this notebook is tuned for a T4. bfloat16 is unavailable on\nTuring GPUs, so float16 is used below; VRAM below 10 GB will OOM.')


In [ ]:
# float16, never bfloat16: a T4 is Turing (sm_75) and has no bf16 support.
from diffusers import AutoencoderKLWan, WanPipeline

MODEL = manifest['model']
print('loading', MODEL, '... please wait, ~9 GB of weights')

vae = AutoencoderKLWan.from_pretrained(
    MODEL, subfolder='vae', torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(
    MODEL, vae=vae, torch_dtype=torch.float16)
pipe.set_progress_bar_config(disable=True)

OFFLOAD = False  # flip to True if you hit CUDA out of memory
if OFFLOAD:
    pipe.enable_model_cpu_offload()
else:
    pipe.to('cuda')

print('ready')


In [ ]:
from diffusers.utils import export_to_video

total = len(jobs)
for n, job in enumerate(jobs, 1):
    out = CLIPS / job['out_name']
    if out.exists() and out.stat().st_size > 20_000:
        print(f'[{n}/{total}] skip {job["out_name"]} (already done)')
        continue

    print(f'[{n}/{total}] {job["id"]} {job["kind"]}  seed={job["seed"]}  '
          f'{job["width"]}x{job["height"]} {job["num_frames"]}f')
    t0 = time.time()
    gen = torch.Generator(device='cuda').manual_seed(int(job['seed']))
    try:
        frames = pipe(
            prompt=job['prompt'],
            negative_prompt=job['negative_prompt'],
            height=int(job['height']),
            width=int(job['width']),
            num_frames=int(job['num_frames']),
            guidance_scale=float(manifest['guidance_scale']),
            num_inference_steps=int(manifest['steps']),
            generator=gen,
        ).frames[0]
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print('  OOM. Re-run with OFFLOAD = True in the previous cell.')
        raise

    export_to_video(frames, str(out), fps=int(job['fps']))
    torch.cuda.empty_cache()
    print(f'  done in {time.time() - t0:.0f}s -> {out.name}')

print('\nall jobs complete')


In [ ]:
missing = [j['out_name'] for j in jobs if not (CLIPS / j['out_name']).exists()]
print(f'{len(jobs) - len(missing)}/{len(jobs)} clips present')
if missing:
    print('MISSING:', ', '.join(missing))
else:
    zip_path = ROOT / 'clips.zip'
    base = str(CLIPS.parent)
    shutil.make_archive(str(zip_path.with_suffix('')), 'zip', base, 'clips')
    print(f'\nzipped -> {zip_path}')
    print('Download it from the Drive panel, unzip over yt_horror/render/clips/wan/,')
    print('then run: python pipeline.py episode 1 --provider wan')
